# Preprocesamiento de la Base de Datos Clínica

In [1]:
import pandas as pd
import numpy as np

def preprocesar_clinica(ruta_archivo):
    try:
        df = pd.read_csv(ruta_archivo)
    except:
        df = pd.read_excel(ruta_archivo)
        
    # Se eliminan columnas y filas vacías
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    df = df.dropna(subset=['DNA_numeric'])
        
    # Corrección de IDs
    if 'DNA_numeric' in df.columns:
        # Se fuerza a texto y se quitan espacios
        df['DNA_numeric'] = df['DNA_numeric'].astype(str).str.strip()
        
        # Se corta por el punto y coma y se mantiene el primer valor
        df['DNA_numeric'] = df['DNA_numeric'].str.split(';').str[0].str.strip()
        
        
    # Limpieza de variables binarias
    cols_binarias = ['TROMBOSIS', 'TROMBOSISRECURRENTE', 'HEMORRAGIA', 'AGENESIAILIACA', 'AGENESIARENAL', 'AGENESIA_CONFIRMADA (No=N, Sí=Sí)', 'ANOMALIASCARDIACAS', 'OTRAS_MALFORMACIONES_CUALQUIERA']
    map_si_no = {'SI': 'SÍ', 'S': 'SÍ', 'Si' : 'SÍ', 'Sí' : 'SÍ', 'SÍ' : 'SÍ', 'NO': 'NO', 'No': 'NO'}
    for col in cols_binarias:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper().map(map_si_no)

    if 'SEXO' in df.columns:
        df['SEXO'] = df['SEXO'].astype(str).str.strip().str.upper().replace({'F': 'MUJER', 'M': 'MUJER', 'f' : 'MUJER', 'H': 'HOMBRE'})
        
    # Corrección edad 
    if 'EDAD' in df.columns:
        df['EDAD'] = df['EDAD'].round().astype('Int64')
        
    if 'EDADTROMBOSIS' in df.columns:
        df['EDADTROMBOSIS'] = df['EDADTROMBOSIS'].round().astype('Int64')
        
    if 'NivelesProteinaS' in df.columns:
        df['NivelesProteinaS'] = pd.to_numeric(df['NivelesProteinaS'].astype(str).str.replace(',', '.'), errors='coerce')

        
    variantes_nulo = ['NONE', 'NAN', 'NULL', '?', 'N/A', '', 'NONE ']
    
    
    df = df.replace(variantes_nulo, np.nan) 
    df['TIPO AGENESIA 3 CATEGORIAS'] = df['TIPO AGENESIA 3 CATEGORIAS'].astype(str).replace(['None', 'none', 'nan', 'nan '], np.nan)
    df = df.replace('nan', np.nan)
    
    return df

# Ejecución
ruta_clinica = "Agenesia_DNA_Phenotypes.xlsx"
df_clinica_limpia = preprocesar_clinica(ruta_clinica)


# Guardar el resultado
df_clinica_limpia.to_csv("BDD_Clinica.csv", index=False)
df_clinica_limpia.to_excel("BDD_Clinica.xlsx", index=False)

print(f"Pacientes totales: {len(df_clinica_limpia)}")

Pacientes totales: 103
